### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [308]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [309]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

### Data generator

##### Support functions

In [310]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [311]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1000, "std_dev": 200, "max_income": 2000},
        (36, 40): {"mean": 1200, "std_dev": 200, "max_income": 2400},
        (41, 45): {"mean": 1500, "std_dev": 250, "max_income": 3000},
        (46, 50): {"mean": 1800, "std_dev": 300, "max_income": 3600},
        (51, 55): {"mean": 2000, "std_dev": 350, "max_income": 4000},
        (56, 60): {"mean": 2200, "std_dev": 400, "max_income": 4300},
        (61, 65): {"mean": 2400, "std_dev": 600, "max_income": 4500},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.6, 1.5), (0.6, 0.65, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [312]:
############################### Function to generate the credit to be requested ##################################
# Note the credits will be exactly for one year
def generate_credit_requested(income): ### It will depend on the income
    # The will maximum enter as for a credit that represent between 50% and 150% of their income and the probabilities of any values are equal, so a uniform distribution
    percentage_of_monthly_income = random.uniform(0.5, 1.5)
    yearly_income = income * 12 # must be changes once dynamically made
    credit_requested_yearly = yearly_income *percentage_of_monthly_income
    credit_requested_monthly = credit_requested_yearly / 12
    percentage_credit_month_income = credit_requested_monthly/income
    return credit_requested_monthly, percentage_credit_month_income
    

In [313]:
############################### Function to generate the whether the person defaults or not ##################################
def generate_default_label(profession, past_credits, debt_to_income_ratio_before_credit, credit_to_income_ratio):
    """This simulates a default label (0/1) based on financial risk factors."""
    """What we will use will be the """
    
    # Configurable risk settings per profession
    risk_settings = {
        "Unemployed_LowSkilled":     {"base": 0.15, "weights": (0.30, 0.6, 0.4)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "LowSkilled":                {"base": 0.08, "weights": (0.20, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_MediumSkilled": {"base": 0.12, "weights": (0.30, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "MediumSkilled":            {"base": 0.05, "weights": (0.20, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_HighSkilled":   {"base": 0.09, "weights": (0.30, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "HighSkilled":              {"base": 0.2,  "weights": (0.20, 0.4, 0.2)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
    }

    # setting variable is sett to retrieve the element of reisk_settings for the profession given
    settings = risk_settings.get(profession)
    # In case there is the profession given for the function does not match any of the professions above listed
    if not settings:
        raise ValueError(f"Unknown profession: {profession}")
    
    # Defining the weigths for the calcualtion of the probability of default
    w1, w2, w3 = settings["weights"]
    # Defining the base probability of default
    base = settings["base"]

    # Calculating the risk factor with the weigths
    risk_factor = past_credits * w1 + debt_to_income_ratio_before_credit * w2 + credit_to_income_ratio * w3
    # Defining the default probability
    default_probability = min(1, base + risk_factor)

    # We want a non-deterministic y-categorical variable that will make that same profiles will not always lead to the same result
    # So even if two people may fall on the same profile, maybe they will not default
    # return 1 if random.random() < default_probability else 0
    return int(random.random() < default_probability)

##### Data Generator for the original state of individuals

In [314]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        
        # Debt to income ratio: PARTIAL, before the credit
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio_partial = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio_partial = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        monthly_credit = generate_credit_requested(monthly_income)[0]
        credit_to_income_ratio = generate_credit_requested(monthly_income)[1]
        
        # calculating the monthly debt if the monthluy credit is issued
        debt_after_credit = savings_debt - monthly_credit
        if debt_after_credit > 0: # if still the monthly savings are higher than the credit, then the total debt to incom ratio should be zero
            debt_to_income_ratio_total = 0
        else: 
            debt_to_income_ratio_total = abs(debt_after_credit/monthly_income)
        
        # ---------------- Y-Variable --------------------------------#
        default_not_default = generate_default_label(profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio)
        
        data.append({
            'name': name, # independent
            'age': age, # independent
            'educational level': education_level, # independent
            'number of not paid past credits': past_credits, # independent
            'dependents': dependents, # independent
            'profession': profession, # depends on education
            'monthly income': monthly_income, # depends on profession and age
            'monthly expenditure': monthly_expenditure, # depends on income, mean income per age and profession, and the number of dependents
            'savings (debt)': savings_debt, # monthly income - monthly expenditure
            'debt-to-income ratio before credit': debt_to_income_ratio_partial, # abs(savings_debt/monthly_income)
            'credit: monthly amount': monthly_credit, # depends on the income
            'credit-to-income ratio': credit_to_income_ratio, # credit/income
            'debt-to-income ratio after credit': debt_to_income_ratio_total, # abs((savings_debt - credit)/monthly_income)
            'y-categorical-default': default_not_default # depending on profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [315]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,debt-to-income ratio after credit,y-categorical-default
0,name0,36,ausbildung,0,0,MediumSkilled,2190.0,2243.855222,-53.855222,0.024591,3018.088724,1.216446,1.402714,0
1,name1,52,ausbildung,1,1,MediumSkilled,2256.0,2994.634998,-738.634998,0.327409,1368.398224,1.198220,0.933969,1
2,name2,39,ausbildung,2,0,MediumSkilled,1599.0,2195.599823,-596.599823,0.373108,1440.265576,1.307998,1.273837,1
3,name3,52,bachelor degree,1,0,HighSkilled,6584.0,15244.086793,-8660.086793,1.315323,7636.940212,1.314361,2.475247,1
4,name4,33,ausbildung,0,3,MediumSkilled,2009.0,1482.939043,526.060957,0.000000,2603.720219,1.408575,1.034176,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,59,bachelor degree,0,0,HighSkilled,8010.0,7034.633884,975.366116,0.000000,5320.011603,0.580895,0.542403,1
996,name996,55,high school or lower,1,0,LowSkilled,1894.0,2339.545778,-445.545778,0.235241,2278.446800,0.897712,1.438222,1
997,name997,31,bachelor degree,0,0,HighSkilled,3274.0,3683.995118,-409.995118,0.125228,2592.083237,1.064708,0.916945,0
998,name998,37,post graduate degree,0,0,HighSkilled,5190.0,2486.317667,2703.682333,0.000000,6939.129420,1.422171,0.816078,0


##### DF statistics

In [316]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,debt-to-income ratio after credit,y-categorical-default
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,44.813000,0.517000,0.536000,3542.356000,3333.370365,208.985635,0.122762,3511.535718,0.989203,0.958072,0.541000
std,9.193833,0.733655,0.795825,2471.948643,2535.122363,1550.322251,0.243266,2782.868639,0.285136,0.448289,0.498566
min,30.000000,0.000000,0.000000,500.000000,299.321418,-9979.616093,0.000000,352.904553,0.500801,0.000000,0.000000
25%,37.000000,0.000000,0.000000,1750.000000,1539.063463,-368.896494,0.000000,1569.343602,0.742441,0.641579,0.000000
50%,45.000000,0.000000,0.000000,2810.000000,2540.770289,195.552302,0.000000,2629.356191,0.995938,0.943996,1.000000
75%,53.000000,1.000000,1.000000,4655.000000,4411.507241,758.598885,0.159005,4614.117003,1.230140,1.215296,1.000000
max,60.000000,3.000000,4.000000,15494.000000,18591.616093,8182.024766,2.043547,21946.308023,1.497397,3.235363,1.000000


Monthly income by profession

In [317]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,355.0,5902.878873,2504.486489,1867.0,3870.00,5552.0,7489.00,15494.0
LowSkilled,254.0,1596.015748,507.419862,542.0,1170.50,1500.5,1955.25,3078.0
MediumSkilled,292.0,2923.787671,1061.973173,1125.0,2059.75,2743.5,3681.50,5840.0
Unemployed_HighSkilled,39.0,3243.589744,1039.495037,1500.0,2250.00,3500.0,4000.00,4500.0
Unemployed_LowSkilled,32.0,703.125000,192.997368,500.0,500.00,600.0,900.00,1100.0
Unemployed_MediumSkilled,28.0,1382.142857,399.122715,900.0,1087.50,1300.0,1750.00,2000.0


Debt-to-income ratio by profession

In [318]:
df_1.groupby('profession')['debt-to-income ratio before credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,355.0,0.137206,0.275249,0.0,0.0,0.000000,0.162022,1.944201
LowSkilled,254.0,0.127328,0.228826,0.0,0.0,0.000000,0.191314,1.271388
MediumSkilled,292.0,0.117051,0.244644,0.0,0.0,0.000000,0.144718,2.043547
Unemployed_HighSkilled,39.0,0.076189,0.124226,0.0,0.0,0.000000,0.143099,0.421715
Unemployed_LowSkilled,32.0,0.088858,0.117747,0.0,0.0,0.016822,0.162349,0.412553
Unemployed_MediumSkilled,28.0,0.061380,0.105985,0.0,0.0,0.000000,0.096787,0.347423


In [319]:
df_1.groupby('profession')['debt-to-income ratio after credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,355.0,0.991290,0.474383,0.000000,0.668487,0.946591,1.239631,2.818685
LowSkilled,254.0,0.948299,0.443487,0.000000,0.622046,0.951034,1.254771,2.410230
MediumSkilled,292.0,0.922228,0.449563,0.000000,0.596533,0.919268,1.178484,3.235363
Unemployed_HighSkilled,39.0,0.949986,0.344514,0.275536,0.771411,0.938876,1.137986,1.773478
Unemployed_LowSkilled,32.0,1.001830,0.365826,0.263082,0.815107,1.012316,1.287644,1.623691
Unemployed_MediumSkilled,28.0,0.960609,0.333914,0.525536,0.694351,0.875576,1.180098,1.770694


In [320]:
df_1.groupby('profession')['credit-to-income ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,355.0,0.968766,0.282939,0.503875,0.715284,0.956246,1.215457,1.495096
LowSkilled,254.0,0.999739,0.271214,0.500801,0.775269,1.020395,1.225665,1.494010
MediumSkilled,292.0,0.987747,0.293299,0.502958,0.732563,0.997205,1.223297,1.497397
Unemployed_HighSkilled,39.0,0.970092,0.303701,0.505006,0.716627,0.939304,1.284109,1.454409
Unemployed_LowSkilled,32.0,1.096768,0.274535,0.519750,0.942547,1.134434,1.343827,1.492364
Unemployed_MediumSkilled,28.0,1.071599,0.313021,0.525144,0.777293,1.083522,1.363109,1.468143


In [321]:
df_1.groupby('profession')['y-categorical-default'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,355.0,0.504225,0.500688,0.0,0.0,1.0,1.0,1.0
LowSkilled,254.0,0.641732,0.480438,0.0,0.0,1.0,1.0,1.0
MediumSkilled,292.0,0.489726,0.500753,0.0,0.0,0.0,1.0,1.0
Unemployed_HighSkilled,39.0,0.512821,0.506370,0.0,0.0,1.0,1.0,1.0
Unemployed_LowSkilled,32.0,0.718750,0.456803,0.0,0.0,1.0,1.0,1.0
Unemployed_MediumSkilled,28.0,0.464286,0.507875,0.0,0.0,0.0,1.0,1.0
